In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA

## Configuration

In [ ]:
DATASET_DIR = "../abstrait-v4/"
FICHIER_CNN_BRUT = "X_cnn_features.npy"
FICHIER_CNN_PCA = "cnn_features_pca.npy"

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
PATCH_SIZE = 256
NUM_PATCHES = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_COMPONENTS_PCA = 50

K_RANGE = range(2, 30)
K_OPTIMAL = None

N_NEIGHBORS_SIMILARITY = 6
N_NEIGHBORS_GRID = 11
NB_EXEMPLAIRES_PAR_CLUSTER = 7
SEUIL_ARTISTES_MAJEURS = 20

print(f"Device : {DEVICE}")
print(f"Dataset : {DATASET_DIR}")

## 1. Modèle VGG16 et matrice de Gram

In [ ]:
def gram_matrix(input_tensor):
    b, c, h, w = input_tensor.size()
    features = input_tensor.view(b, c, h * w)
    G = torch.bmm(features, features.transpose(1, 2))
    return G.div(c * h * w)


class VGGStyleExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.slice1 = nn.Sequential(*list(vgg.children())[:4])
        self.slice2 = nn.Sequential(*list(vgg.children())[4:9])
        self.slice3 = nn.Sequential(*list(vgg.children())[9:16])
        self.slice4 = nn.Sequential(*list(vgg.children())[16:23])
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):
        features = []
        x = self.slice1(x); features.append(x)
        x = self.slice2(x); features.append(x)
        x = self.slice3(x); features.append(x)
        x = self.slice4(x); features.append(x)
        return features

## 2. Extraction de la signature stylistique

In [ ]:
def extract_painting_signature(image_path, model, device, patch_size=256, num_patches=10):
    preprocess = transforms.Compose([
        transforms.RandomCrop(patch_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    img = Image.open(image_path).convert('RGB')
    if img.size[0] < patch_size or img.size[1] < patch_size:
        img = img.resize((patch_size, patch_size))

    all_patch_vectors = []
    for _ in range(num_patches):
        patch_tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            layer_features = model(patch_tensor)
            patch_style_vector = []
            for feat in layer_features:
                G = gram_matrix(feat)
                triu_indices = torch.triu_indices(G.size(1), G.size(2))
                flat_G = G[0, triu_indices[0], triu_indices[1]]
                patch_style_vector.append(flat_G)
            full_patch_vector = torch.cat(patch_style_vector)
            all_patch_vectors.append(full_patch_vector)

    stacked_vectors = torch.stack(all_patch_vectors)
    final_painting_vector = torch.mean(stacked_vectors, dim=0)
    return final_painting_vector.cpu().numpy()

## 3. Initialisation du modèle et test

In [ ]:
style_model = VGGStyleExtractor().to(DEVICE)
style_model.eval()

image_test_path = os.path.join(DATASET_DIR, "afro_1.jpg")


vecteur_style = extract_painting_signature(image_test_path, style_model, DEVICE)
print("Extraction réussie !")
print(f"Dimension du vecteur CNN extrait : {vecteur_style.shape}")


## 4. Extraction des features sur le dataset complet

In [ ]:
features_cnn_list = []
image_names = []

all_images = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(IMAGE_EXTENSIONS)]
print(f"Début de l'extraction sur {len(all_images)} images...")

for img_name in tqdm(all_images):
    img_path = os.path.join(DATASET_DIR, img_name)
    vec = extract_painting_signature(img_path, style_model, DEVICE, PATCH_SIZE, NUM_PATCHES)
    features_cnn_list.append(vec)
    image_names.append(img_name)

X_cnn = np.array(features_cnn_list)
np.save(FICHIER_CNN_BRUT, X_cnn)

## 5. Normalisation et réduction PCA

In [ ]:
if os.path.exists(FICHIER_CNN_BRUT):
    print(f"Chargement des caractéristiques depuis '{FICHIER_CNN_BRUT}'...")
    X_cnn = np.load(FICHIER_CNN_BRUT)
    print(f"Matrice X_cnn chargée avec succès. Dimensions : {X_cnn.shape}")
else:
    raise FileNotFoundError(f"Le fichier {FICHIER_CNN_BRUT} est introuvable. Veuillez relancer l'extraction (étape 4).")


def reduce_and_scale_block_robust(X_block, n_components):
    X_clean = np.nan_to_num(X_block, nan=np.nanmedian(X_block))
    l2_normalizer = Normalizer(norm='l2')
    X_l2 = l2_normalizer.fit_transform(X_clean)
    pca = PCA(n_components=n_components)
    X_reduced = pca.fit_transform(X_l2)
    variance_retenue = sum(pca.explained_variance_ratio_) * 100
    print(f"Variance expliquée par les {n_components} composantes : {variance_retenue:.2f}%")
    scaler = StandardScaler()
    X_final = scaler.fit_transform(X_reduced)
    return X_final


print("Application de la réduction robuste sur le bloc CNN...")
X_final_scaled = reduce_and_scale_block_robust(X_cnn, n_components=N_COMPONENTS_PCA)
print(f"Matrice corrigée prête ! Nouvelles dimensions : {X_final_scaled.shape}")

np.save(FICHIER_CNN_PCA, X_final_scaled)
print(f"Features CNN réduites sauvegardées sous '{FICHIER_CNN_PCA}'.")

## 6. Chargement des features réduites

In [ ]:
if os.path.exists(FICHIER_CNN_PCA):
    X_cnn = np.load(FICHIER_CNN_PCA)
    print(f"Matrice CNN chargée. Dimensions de l'espace latent (VGG16 + PCA) : {X_cnn.shape}")
else:
    print(f"Erreur : Le fichier '{FICHIER_CNN_PCA}' est introuvable.")

## 7. Recherche du K optimal (Coude + Silhouette)

In [ ]:
K_range = K_RANGE
inertia_values = []
silhouette_scores = []

print("Calcul des métriques pour trouver le K optimal...")

for k in tqdm(K_range):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels_temp = kmeans.fit_predict(X_final_scaled)
    inertia_values.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_final_scaled, cluster_labels_temp))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(K_range, inertia_values, marker='o', linestyle='-', color='b', linewidth=2)
ax1.set_title("Méthode du Coude (Inertie)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Nombre de clusters (K)")
ax1.set_ylabel("Inertie intra-cluster")
ax1.set_xticks(K_range)
ax1.grid(True, linestyle='--', alpha=0.7)

ax2.plot(K_range, silhouette_scores, marker='s', linestyle='-', color='g', linewidth=2)
ax2.set_title("Score de Silhouette", fontsize=14, fontweight='bold')
ax2.set_xlabel("Nombre de clusters (K)")
ax2.set_ylabel("Score (Proche de 1 = Meilleur)")
ax2.set_xticks(K_range)
ax2.grid(True, linestyle='--', alpha=0.7)

K_OPTIMAL = list(K_range)[silhouette_scores.index(max(silhouette_scores))]
ax2.axvline(x=K_OPTIMAL, color='r', linestyle='--', label=f'Max Silhouette (K={K_OPTIMAL})')
ax2.legend()

plt.tight_layout()
print(f"=> K optimal (meilleure silhouette) : K = {K_OPTIMAL}")

## 8. Valeur de K

In [ ]:
K_OPTIMAL = 15

## 9. Clustering K-Means final

In [ ]:
k_choisi = K_OPTIMAL

kmeans = KMeans(n_clusters=k_choisi, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_final_scaled)

## 10. Recherche de similarité (1 image aléatoire)

In [ ]:
if 'image_names' not in locals():
    image_names = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(IMAGE_EXTENSIONS)]
    print(f"Liste recréée : {len(image_names)} images trouvées.")

if len(image_names) != X_final_scaled.shape[0]:
    print("ATTENTION : Le nombre d'images ne correspond pas à la taille de la matrice CNN !")

nn_model = NearestNeighbors(n_neighbors=N_NEIGHBORS_SIMILARITY + 1, metric='cosine')
nn_model.fit(X_final_scaled)

idx_random = random.randint(0, len(image_names) - 1)
target_vector = X_final_scaled[idx_random].reshape(1, -1)
target_name = image_names[idx_random]

distances, indices = nn_model.kneighbors(target_vector)

fig, axes = plt.subplots(1, N_NEIGHBORS_SIMILARITY + 1, figsize=(20, 5))
fig.suptitle(f"Recherche de similarité pour : {target_name.split('_')[0].capitalize()}", fontsize=16, fontweight='bold')

for i, ax in enumerate(axes):
    voisin_idx = indices[0][i]
    voisin_name = image_names[voisin_idx]
    voisin_artiste = voisin_name.split('_')[0].capitalize()
    img_path = os.path.join(DATASET_DIR, voisin_name)
    try:
        img = mpimg.imread(img_path)
        ax.imshow(img)
        if i == 0:
            ax.set_title(f"CIBLE\n{voisin_artiste}", color='blue', fontweight='bold')
            for spine in ax.spines.values():
                spine.set_edgecolor('blue')
                spine.set_linewidth(3)
        else:
            ax.set_title(f"{voisin_artiste}\n(Dist: {distances[0][i]:.2f})")
    except FileNotFoundError:
        ax.set_title(f"Image introuvable:\n{voisin_name}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 11. Visualisation t-SNE 2D

In [ ]:
print("Calcul du t-SNE en 2D...")
tsne = TSNE(n_components=2, perplexity=40, random_state=42)
X_tsne_2d = tsne.fit_transform(X_final_scaled)

kmeans = KMeans(n_clusters=k_choisi, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_final_scaled)

plt.figure(figsize=(14, 10))
scatter = plt.scatter(
    X_tsne_2d[:, 0], X_tsne_2d[:, 1],
    c=cluster_labels, cmap='viridis', alpha=0.7, edgecolors='w', s=40
)
plt.title("Visualisation 2D de l'espace des styles (VGG16 + t-SNE)", fontsize=16, fontweight='bold')
plt.xlabel("Dimension Latente 1")
plt.ylabel("Dimension Latente 2")
plt.colorbar(scatter, label=f"Cluster attribué (K={k_choisi})")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 12. Grille de similarité (multi-cibles)

In [ ]:
NUM_TARGETS = 5
N_NEIGHBORS = 6
N_COLS = N_NEIGHBORS + 1

nn_model = NearestNeighbors(n_neighbors=N_COLS, metric='cosine')
nn_model.fit(X_final_scaled)

random_indices = random.sample(range(len(image_names)), NUM_TARGETS)

fig, axes = plt.subplots(NUM_TARGETS, N_COLS, figsize=(28, 22))
fig.suptitle(
    f"Analyse des styles : {NUM_TARGETS} Cibles et leurs {N_NEIGHBORS} Plus Proches Voisins (Features CNN)",
    fontsize=22, fontweight='bold', y=0.98
)

for row, idx in enumerate(random_indices):
    target_vector = X_final_scaled[idx].reshape(1, -1)
    distances, indices = nn_model.kneighbors(target_vector)

    for col in range(N_COLS):
        ax = axes[row, col]
        voisin_idx = indices[0][col]
        voisin_name = image_names[voisin_idx]
        voisin_artiste = voisin_name.split('_')[0].capitalize()
        img_path = os.path.join(DATASET_DIR, voisin_name)
        try:
            img = mpimg.imread(img_path)
            ax.imshow(img)
            if col == 0:
                ax.set_title(f"CIBLE\n{voisin_artiste}", color='darkblue', fontweight='bold', fontsize=12)
                for spine in ax.spines.values():
                    spine.set_color('darkblue')
                    spine.set_linewidth(4)
            else:
                ax.set_title(f"{voisin_artiste}\n(d={distances[0][col]:.2f})", fontsize=10)
                for spine in ax.spines.values():
                    spine.set_color('gray')
                    spine.set_linewidth(1)
        except FileNotFoundError:
            ax.set_title("Image absente", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.subplots_adjust(top=0.93, hspace=0.4)
plt.show()

## 13. Profilage visuel des clusters

In [ ]:
nb_exemples_par_cluster = NB_EXEMPLAIRES_PAR_CLUSTER

fig, axes = plt.subplots(k_choisi, nb_exemples_par_cluster, figsize=(20, 3 * k_choisi))
fig.suptitle("Profilage Visuel des Clusters", fontsize=18, fontweight='bold', y=0.98)

for cluster_id in range(k_choisi):
    indices_cluster = np.where(cluster_labels == cluster_id)[0]
    nb_samples = min(nb_exemples_par_cluster, len(indices_cluster))
    echantillon_indices = random.sample(list(indices_cluster), nb_samples)

    for col in range(nb_exemples_par_cluster):
        axes[cluster_id, col].axis('off')

    for col, idx in enumerate(echantillon_indices):
        ax = axes[cluster_id, col]
        nom_image = image_names[idx]
        artiste = nom_image.split('_')[0].capitalize()
        img_path = os.path.join(DATASET_DIR, nom_image)
        try:
            img = mpimg.imread(img_path)
            ax.imshow(img)
            ax.set_title(f"{artiste}", fontsize=10)
        except FileNotFoundError:
            ax.set_title("Introuvable", fontsize=10)

    axes[cluster_id, 0].text(
        -0.1, 0.5, f"Cluster {cluster_id}\n({len(indices_cluster)} œuvres)",
        transform=axes[cluster_id, 0].transAxes,
        fontsize=14, fontweight='bold', va='center', ha='right'
    )

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

## 14. Heatmap artistes × clusters

In [ ]:
artistes = [nom.split('_')[0].capitalize() for nom in image_names]
df_results = pd.DataFrame({'Artiste': artistes, 'Cluster': cluster_labels})

comptage_artistes = df_results['Artiste'].value_counts()
artistes_majeurs = comptage_artistes[comptage_artistes >= SEUIL_ARTISTES_MAJEURS].index
df_majeurs = df_results[df_results['Artiste'].isin(artistes_majeurs)]

crosstab = pd.crosstab(df_majeurs['Artiste'], df_majeurs['Cluster'])
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

plt.figure(figsize=(14, 10))
sns.heatmap(crosstab_pct, annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={'label': "% des œuvres de l'artiste"})
plt.title("Répartition stylistique : Pourcentage de l'œuvre d'un artiste par cluster",
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel("ID du Cluster (K-Means)", fontsize=12)
plt.ylabel("Artistes Majeurs", fontsize=12)
plt.tight_layout()
plt.show()

## 15. Rapport JSON

In [ ]:
artistes = [nom.split('_')[0].capitalize() for nom in image_names]
df_results = pd.DataFrame({'Artiste': artistes, 'Cluster': cluster_labels})

tailles_clusters = df_results['Cluster'].value_counts().sort_index().to_dict()
score_sil = silhouette_score(X_final_scaled, cluster_labels)

top_artistes = df_results['Artiste'].value_counts().head(8).index
df_top = df_results[df_results['Artiste'].isin(top_artistes)]
crosstab = pd.crosstab(df_top['Artiste'], df_top['Cluster'])

rapport_ia = {
    "1_Configuration": {
        "Nombre_de_Clusters_K": int(len(tailles_clusters)),
        "Score_de_Silhouette": round(float(score_sil), 4)
    },
    "2_Equilibre_des_Clusters": {f"Cluster_{k}": int(v) for k, v in tailles_clusters.items()},
    "3_Repartition_Top_Artistes": crosstab.to_dict(orient="index")
}

print("=" * 60)
print(json.dumps(rapport_ia, indent=4, ensure_ascii=False))
print("=" * 60)